In [0]:
# ================================================================
# FINGUARD HIGH-VALUE TRANSACTION EMAIL NOTIFICATION
#
# IMPORTANT:
# Run this as a normal Databricks Notebook Task.
# DO NOT add this notebook as Declarative Pipeline source code.
# ================================================================

import smtplib

from datetime import datetime, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from html import escape

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)


# ================================================================
# 1. CONFIGURATION
# ================================================================

ALERT_TABLE = "finguard.gold.high_value_transactions_alert"

EMAIL_LOG_TABLE = "finguard.gold.email_alerts_sent"

SENDER_EMAIL = "phemanthphanikumar@gmail.com"

# All alerts go here while TEST_MODE = True
TEST_RECIPIENT_EMAIL = "phemanthphanikumar@gmail.com"

# Keep True during testing.
# Change to False when you want emails sent to customer_email.
TEST_MODE = True

EMAIL_SUBJECT = "FinGuard Alert: High Value Transaction"


# ================================================================
# 2. GET GMAIL APP PASSWORD
# ================================================================

GMAIL_APP_PASSWORD = dbutils.secrets.get(
    scope="finguard-scope",
    key="gmail_api_key"
)

if (
    GMAIL_APP_PASSWORD is None
    or GMAIL_APP_PASSWORD.strip() == ""
):
    raise Exception(
        "Gmail App Password is empty. "
        "Check secret scope 'finguard-scope' "
        "and key 'gmail_api_key'."
    )

GMAIL_APP_PASSWORD = GMAIL_APP_PASSWORD.strip()

print("Gmail credential retrieved successfully.")


# ================================================================
# 3. HELPER FUNCTION
# ================================================================

def safe_value(
    row,
    field_name,
    default_value="Not available"
):

    row_dict = row.asDict()

    if field_name not in row_dict:
        return default_value

    value = row_dict[field_name]

    if value is None:
        return default_value

    return escape(str(value))


# ================================================================
# 4. EMAIL BODY
# ================================================================

def build_alert_email_body(
    row,
    actual_recipient
):

    customer_name = safe_value(
        row,
        "customer_name",
        "Customer"
    )

    customer_id = safe_value(
        row,
        "customer_id"
    )

    transaction_id = safe_value(
        row,
        "transaction_id"
    )

    transaction_amount = safe_value(
        row,
        "transaction_amount"
    )

    transaction_limit = safe_value(
        row,
        "transaction_limit"
    )

    amount_over_limit = safe_value(
        row,
        "amount_over_limit"
    )

    currency = safe_value(
        row,
        "currency",
        ""
    )

    merchant_name = safe_value(
        row,
        "merchant_name"
    )

    transaction_timestamp = safe_value(
        row,
        "transaction_timestamp"
    )

    transaction_type = safe_value(
        row,
        "transaction_type"
    )

    payment_channel = safe_value(
        row,
        "payment_channel"
    )

    city = safe_value(
        row,
        "city"
    )

    country = safe_value(
        row,
        "country"
    )

    customer_email = safe_value(
        row,
        "customer_email"
    )

    testing_section = ""

    if TEST_MODE:

        testing_section = f"""
        <div style="
             background-color:#fff3cd;
             padding:10px;
             margin-top:15px;
             border:1px solid #ffeeba;">

            <strong>TEST MODE</strong><br>
            Customer email: {customer_email}<br>
            Actual recipient: {escape(actual_recipient)}

        </div>
        """

    return f"""
    <html>

    <body style="
        font-family:Arial,sans-serif;
        color:#222222;
    ">

        <h2 style="color:#c62828;">
            High Value Transaction Alert
        </h2>

        <p>
            Dear {customer_name},
        </p>

        <p>
            A transaction on your account exceeded your
            configured transaction limit.
        </p>

        <table
            border="1"
            cellpadding="8"
            cellspacing="0"
            style="border-collapse:collapse;"
        >

            <tr>
                <th align="left">Customer ID</th>
                <td>{customer_id}</td>
            </tr>

            <tr>
                <th align="left">Transaction ID</th>
                <td>{transaction_id}</td>
            </tr>

            <tr>
                <th align="left">Transaction Amount</th>
                <td>
                    {transaction_amount} {currency}
                </td>
            </tr>

            <tr>
                <th align="left">Transaction Limit</th>
                <td>
                    {transaction_limit} {currency}
                </td>
            </tr>

            <tr>
                <th align="left">Amount Over Limit</th>
                <td>
                    {amount_over_limit} {currency}
                </td>
            </tr>

            <tr>
                <th align="left">Merchant</th>
                <td>{merchant_name}</td>
            </tr>

            <tr>
                <th align="left">Date & Time</th>
                <td>{transaction_timestamp}</td>
            </tr>

            <tr>
                <th align="left">Transaction Type</th>
                <td>{transaction_type}</td>
            </tr>

            <tr>
                <th align="left">Payment Channel</th>
                <td>{payment_channel}</td>
            </tr>

            <tr>
                <th align="left">Location</th>
                <td>
                    {city}, {country}
                </td>
            </tr>

        </table>

        <p>
            If you did not make this transaction,
            please contact FinGuard Support immediately.
        </p>

        {testing_section}

        <p>
            Thanks,<br>
            <strong>FinGuard Support Team</strong>
        </p>

    </body>

    </html>
    """


# ================================================================
# 5. SEND EMAIL USING GMAIL SMTP
# ================================================================

def send_alert_email(
    recipient_email,
    subject,
    html_body
):

    message = MIMEMultipart("alternative")

    message["From"] = SENDER_EMAIL
    message["To"] = recipient_email
    message["Subject"] = subject

    plain_text = """
FinGuard High Value Transaction Alert

A transaction exceeded the configured transaction limit.

Please review the HTML version of this email for details.

FinGuard Support Team
"""

    message.attach(
        MIMEText(
            plain_text,
            "plain",
            "utf-8"
        )
    )

    message.attach(
        MIMEText(
            html_body,
            "html",
            "utf-8"
        )
    )

    print(
        f"Connecting to Gmail SMTP for {recipient_email}..."
    )

    with smtplib.SMTP(
        "smtp.gmail.com",
        587,
        timeout=60
    ) as smtp_server:

        smtp_server.ehlo()

        print("Starting TLS...")

        smtp_server.starttls()

        smtp_server.ehlo()

        print("Authenticating with Gmail...")

        smtp_server.login(
            SENDER_EMAIL,
            GMAIL_APP_PASSWORD
        )

        print("Gmail authentication successful.")

        smtp_server.send_message(
            message
        )

        print(
            f"Gmail accepted message for: "
            f"{recipient_email}"
        )


# ================================================================
# 6. VALIDATE GOLD ALERT TABLE
# ================================================================

if not spark.catalog.tableExists(ALERT_TABLE):

    raise Exception(
        f"Alert table does not exist: {ALERT_TABLE}. "
        "Run the FinGuard pipeline first."
    )


alerts_source_df = spark.read.table(
    ALERT_TABLE
)

print("")
print("Gold alert table found.")
print("Gold columns:")
print(alerts_source_df.columns)


required_columns = [
    "alert_id",
    "customer_id",
    "customer_name",
    "customer_email",
    "transaction_id",
    "transaction_amount",
    "transaction_limit",
    "amount_over_limit",
    "transaction_timestamp",
    "merchant_name",
    "currency",
    "transaction_type",
    "payment_channel",
    "country",
    "city"
]


missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in alerts_source_df.columns
]


if missing_columns:

    raise Exception(
        "Gold alert table is missing columns: "
        + ", ".join(missing_columns)
        + ". Available columns: "
        + ", ".join(alerts_source_df.columns)
    )


# ================================================================
# 7. CREATE / VALIDATE EMAIL LOG TABLE
# ================================================================

email_log_schema = StructType([

    StructField(
        "alert_id",
        StringType(),
        False
    ),

    StructField(
        "recipient_email",
        StringType(),
        False
    ),

    StructField(
        "customer_email",
        StringType(),
        True
    ),

    StructField(
        "sent_at",
        TimestampType(),
        False
    )

])


expected_log_columns = {
    "alert_id",
    "recipient_email",
    "customer_email",
    "sent_at"
}


if spark.catalog.tableExists(
    EMAIL_LOG_TABLE
):

    existing_log_columns = set(
        spark.read
        .table(EMAIL_LOG_TABLE)
        .columns
    )

    print("")
    print(
        f"Email log table found: "
        f"{EMAIL_LOG_TABLE}"
    )

    print(
        f"Email log columns: "
        f"{sorted(existing_log_columns)}"
    )

    missing_log_columns = (
        expected_log_columns
        - existing_log_columns
    )

    if missing_log_columns:

        raise Exception(
            "Existing email log table has an "
            "incorrect schema. Missing columns: "
            + ", ".join(missing_log_columns)
            + ". Since you are testing, drop "
            + EMAIL_LOG_TABLE
            + " and rerun the notebook."
        )


else:

    empty_log_df = spark.createDataFrame(
        [],
        schema=email_log_schema
    )

    (
        empty_log_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            EMAIL_LOG_TABLE
        )
    )

    print(
        f"Created email log table: "
        f"{EMAIL_LOG_TABLE}"
    )


# ================================================================
# 8. PREPARE GOLD ALERTS
# ================================================================

alerts_df = (

    alerts_source_df

    .select(

        "alert_id",
        "customer_id",
        "customer_name",
        "customer_email",
        "transaction_id",
        "transaction_amount",
        "transaction_limit",
        "amount_over_limit",
        "transaction_timestamp",
        "merchant_name",
        "currency",
        "transaction_type",
        "payment_channel",
        "country",
        "city"

    )

    .filter(

        F.col("alert_id").isNotNull()

        &

        F.col("transaction_id").isNotNull()

    )

    .dropDuplicates(
        ["alert_id"]
    )

)


total_alert_count = alerts_df.count()

print("")
print(
    f"Total alerts in Gold table: "
    f"{total_alert_count}"
)


# ================================================================
# 9. CHECK WHICH ALERTS WERE ALREADY EMAILED
# ================================================================

already_sent_df = (

    spark.read
    .table(EMAIL_LOG_TABLE)

    .select(
        "alert_id"
    )

    .filter(
        F.col("alert_id").isNotNull()
    )

    .dropDuplicates(
        ["alert_id"]
    )

)


new_alerts_df = (

    alerts_df

    .join(
        already_sent_df,
        on="alert_id",
        how="left_anti"
    )

)


new_alert_count = (
    new_alerts_df.count()
)


print(
    f"New alerts waiting for email: "
    f"{new_alert_count}"
)


# ================================================================
# 10. NOTHING TO SEND
# ================================================================

if new_alert_count == 0:

    print("")
    print(
        "There are no new unsent alerts."
    )

    print(
        "If you are testing an alert that was "
        "already emailed, its alert_id is already "
        "present in the email log."
    )


# ================================================================
# 11. SEND NEW ALERT EMAILS
# ================================================================

sent_rows = []

failed_rows = []


for row in new_alerts_df.toLocalIterator():

    alert_id = row["alert_id"]

    customer_email = row["customer_email"]


    # ------------------------------------------------------------
    # Determine recipient
    # ------------------------------------------------------------

    if TEST_MODE:

        recipient_email = (
            TEST_RECIPIENT_EMAIL
        )

    else:

        recipient_email = (
            customer_email
        )


    # ------------------------------------------------------------
    # Validate recipient
    # ------------------------------------------------------------

    if (
        recipient_email is None
        or
        recipient_email.strip() == ""
    ):

        error_message = (
            "Recipient email is missing."
        )

        failed_rows.append({

            "alert_id":
                alert_id,

            "error":
                error_message

        })

        print(
            f"SKIPPED: {alert_id} - "
            f"{error_message}"
        )

        continue


    recipient_email = (
        recipient_email.strip()
    )


    # ------------------------------------------------------------
    # Build email
    # ------------------------------------------------------------

    html_body = (
        build_alert_email_body(
            row,
            recipient_email
        )
    )


    # ------------------------------------------------------------
    # Send email
    # ------------------------------------------------------------

    try:

        print("")
        print(
            f"Sending alert: {alert_id}"
        )

        print(
            f"Recipient: {recipient_email}"
        )


        send_alert_email(

            recipient_email,

            EMAIL_SUBJECT,

            html_body

        )


        sent_rows.append({

            "alert_id":
                alert_id,

            "recipient_email":
                recipient_email,

            "customer_email":
                customer_email,

            "sent_at":
                datetime.now(
                    timezone.utc
                ).replace(
                    tzinfo=None
                )

        })


        print(
            f"SUCCESS: {alert_id}"
        )


    except Exception as error:

        error_message = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )


        failed_rows.append({

            "alert_id":
                alert_id,

            "error":
                error_message

        })


        print(
            f"FAILED: {alert_id}"
        )

        print(
            f"Error: {error_message}"
        )


# ================================================================
# 12. WRITE SUCCESSFUL EMAILS TO LOG
# ================================================================

if sent_rows:

    sent_email_df = (
        spark.createDataFrame(
            sent_rows,
            schema=email_log_schema
        )
    )


    (
        sent_email_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(
            EMAIL_LOG_TABLE
        )
    )


    print("")
    print(
        f"Logged {len(sent_rows)} "
        f"successful email(s)."
    )


# ================================================================
# 13. FINAL RESULT
# ================================================================

print("")
print("==============================================")
print("FINGUARD EMAIL NOTIFICATION RESULT")
print("==============================================")

print(
    f"Test mode              : {TEST_MODE}"
)

print(
    f"Gold alerts            : {total_alert_count}"
)

print(
    f"New alerts             : {new_alert_count}"
)

print(
    f"Emails sent            : {len(sent_rows)}"
)

print(
    f"Failed / skipped       : {len(failed_rows)}"
)


if TEST_MODE:

    print(
        f"Test recipient         : "
        f"{TEST_RECIPIENT_EMAIL}"
    )


if failed_rows:

    print("")
    print("FAILED ALERTS")
    print("----------------------------------------------")

    for failure in failed_rows:

        print(
            f"{failure['alert_id']} -> "
            f"{failure['error']}"
        )


print("==============================================")